In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing

In [47]:
test_labels = np.load('data_lab5/data/test_labels.npy').astype(np.float64)
test_sentences = np.load('data_lab5/data/test_sentences.npy', allow_pickle = True)
train_labels = np.load('data_lab5/data/training_labels.npy').astype(np.float64)
train_sentences = np.load('data_lab5/data/training_sentences.npy', allow_pickle = True)

print (test_sentences.shape)
print (train_sentences.shape)

from numpy import linalg as LA

def normalize_data(train_data, test_data, type = None):
    scaled_train_data = train_data
    scaled_test_data = test_data
    if type == 'standard':
        scaler = preprocessing.StandardScaler()
        scaled_train_data = scaler.transform(train_data)
        scaled_test_data = scaler.transform(test_data)
    elif type == 'l1':
        norm_train = np.sum(np.abs(train_data), axis = 1, keepdims = True)
        norm_train[norm_train == 0] = 1  # prevent division by zero
        scaled_train_data = train_data / norm_train
        
        norm_test = np.sum(np.abs(test_data), axis = 1, keepdims = True)
        norm_test[norm_test == 0] = 1
        scaled_test_data = test_data / norm_test
    elif type == 'l2':
        norm_train = np.sqrt(np.sum(train_data ** 2, axis = 1, keepdims = True))
        norm_train[norm_train == 0] = 1  # prevent division by zero
        scaled_train_data = train_data / norm_train
        
        norm_test = np.sqrt(np.sum(test_data ** 2, axis = 1, keepdims = True))
        norm_test[norm_test == 0] = 1
        scaled_test_data = test_data / norm_test
        
    return scaled_train_data, scaled_test_data

(1840,)
(3734,)


In [9]:
class BagOfWords:
    def __init__ (self):
        self.vocab = {}

    # data este lista de mesaje (lista de liste de strings)
    def build_vocabulary(self, data):
        word_list = []
        word_id = 0
        for mes in data:
            for w in mes:
                if w not in self.vocab:
                    self.vocab[w] = word_id
                    word_id += 1
                    word_list.append(w)
        return word_list

    # data = lista de mesaje de dimensiunea num_samples (lista de liste de strings)
    def get_features(self, data):
        features = np.zeros((len(data), len(self.vocab)))
        #features = [[0 for col in range(len(self.vocab))] for lin in range(len(data))]
        for sample_idx, sample in enumerate(data):
            for word in self.vocab: # self.vocab[word] = word_id
                #print(sample_idx, self.vocab[word])
                if word in self.vocab:
                    features[sample_idx][self.vocab[word]] = sample.count(word)
        return features

    
                
                
        
        

In [11]:
bow = BagOfWords()
vocabulary = bow.build_vocabulary(train_sentences)
train_words_features = bow.get_features(train_sentences)

print(len(bow.vocab))




9522


In [49]:
test_words_features = bow.get_features(test_sentences)

normalized_train_features, normalized_test_features = normalize_data(train_words_features, test_words_features, type = 'l2')

print(normalized_train_features.shape)
print(normalized_test_features.shape)

(3734, 9522)
(1840, 9522)


In [51]:

from sklearn import svm, metrics
svm_model = svm.SVC(C = 1.0, kernel = 'linear', gamma = 'auto')

svm_model.fit(normalized_train_features, train_labels)
y_pred = svm_model.predict(normalized_test_features)

f1_score = metrics.f1_score(test_labels, y_pred, average = 'binary')
print(f'F1 score = {f1_score}')

acc = svm_model.score(normalized_test_features, test_labels)
print(f'Accuracy: {acc}')






F1 score = 0.9409368635437881
Accuracy: 0.9842391304347826
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [53]:
print(np.unique(test_labels))
print(np.unique(y_pred))

# Get the coefficients (weights) of the trained linear SVM
weights = svm_model.coef_[0]  # Since binary classification, coef_ has shape (1, n_features)

# Map vocab: word -> index (reverse to index -> word)
idx_to_word = {idx: word for word, idx in bow.vocab.items()}

# Get top N most spammy words (highest weights)
top_n = 10
top_spam_indices = np.argsort(weights)[-top_n:]
top_spam_words = [(idx_to_word[i], weights[i]) for i in reversed(top_spam_indices)]

# Get top N most non-spammy words (lowest weights)
top_nonspam_indices = np.argsort(weights)[:top_n]
top_nonspam_words = [(idx_to_word[i], weights[i]) for i in top_nonspam_indices]

print("Most spammy words:")
for word, weight in top_spam_words:
    print(f"{word}: {weight:.4f}")

print("\nMost non-spammy words:")
for word, weight in top_nonspam_words:
    print(f"{word}: {weight:.4f}")


[0. 1.]
[0. 1.]
Most spammy words:
STOP: 2.5110
Txt: 2.5104
Call: 2.2822
&: 2.0474
txt: 2.0005
FREE: 1.9378
CALL: 1.8359
mobile: 1.8222
To: 1.7199
Text: 1.6860

Most non-spammy words:
&lt#&gt: -1.5206
me: -1.1549
i: -1.0480
Going: -0.9130
him: -0.9103
Ok: -0.9082
I: -0.9013
Ill: -0.8841
my: -0.8469
Im: -0.8446
